# 16 — Evaluations (Vertex AI Gen AI Evaluation Service)

**AWS-side equivalent:** Bedrock Online Evaluation + custom evaluators
(`notebooks/aws/16-evaluations.ipynb`)
**GCP service:** Vertex AI Gen AI Evaluation Service —
`vertexai.evaluation.{EvalTask, PointwiseMetric, PairwiseMetric, MetricPromptTemplateExamples}`.

_As of 2026-05: the stable import path is `vertexai.evaluation` (the older
`vertexai.preview.evaluation` is still aliased but deprecated in 1.71+). Install with
`pip install google-cloud-aiplatform[evaluation]`. The judge model defaults to
`gemini-2.5-flash` — we keep that, with `gemini-2.5-pro` for reasoning under test._

Flow mirrors the AWS notebook:
1. Build a small EvalDataset (5 prompts + reference answers).
2. Pointwise eval on the candidate model (groundedness + helpfulness).
3. Pairwise SxS between two model versions.
4. Persist the run id into `config.json`.

## Step 1 — Load config and init Vertex AI

In [ ]:
import json
import time
from pathlib import Path

import vertexai

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
STAGING_BUCKET = gcp.get('staging_bucket')
PREFIX = gcp.get('prefix', 'orgiq')
REASONING_MODEL = gcp.get('reasoning_model', 'gemini-2.5-pro')
FAST_MODEL = gcp.get('fast_model', 'gemini-2.5-flash')
BASELINE_MODEL = gcp.get('baseline_model', 'gemini-2.5-flash')

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
assert STAGING_BUCKET, 'gcp.staging_bucket required — eval writes artefacts to GCS.'

vertexai.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)
print(f'✓ Vertex AI initialised — project={PROJECT_ID} region={REGION}')
print(f'  candidate={REASONING_MODEL}  baseline={BASELINE_MODEL}  judge=gemini-2.5-flash')

## Step 2 — Build the EvalDataset

Five hand-crafted OrgIQ cases that mirror the AWS notebook's `harness_cases`. Each row has
`prompt`, `reference` (the gold answer), `context` (graph snippets that the candidate
should ground in), and a `human_score` placeholder. The Evaluation Service builds a
DataFrame from this dict.

Drop in `data/org/harness_cases.json` later if you want to swap from hardcoded to file-based.

In [ ]:
import pandas as pd

eval_cases = [
    {
        'prompt': 'Who is the top Kubernetes expert at the company?',
        'reference': 'Alice Chen is the top Kubernetes expert with composite score 0.74.',
        'context': 'Alice Chen (p_alice) composite_score=0.74, has_skill Kubernetes weight=0.92.',
        'human_score': 5,
    },
    {
        'prompt': 'Tell me about Eve Johnson\'s expertise.',
        'reference': 'Eve Johnson is a security engineer with Authentication (0.95) and Microservices (0.6).',
        'context': 'Eve Johnson (p_eve) title=Security Engineer; skills: Authentication 0.95, Microservices 0.6.',
        'human_score': 5,
    },
    {
        'prompt': 'Who can help with CI/CD?',
        'reference': 'Dave Park has strong CI/CD expertise (0.9). Alice Chen also knows CI/CD (0.8).',
        'context': 'Dave Park (p_dave) has_skill CI/CD weight=0.9; Alice Chen has_skill CI/CD weight=0.8.',
        'human_score': 4,
    },
    {
        'prompt': 'What documents discuss microservices architecture?',
        'reference': 'Service Mesh Patterns authored by Carol Diaz.',
        'context': 'Document "Service Mesh Patterns" authored by Carol Diaz (p_carol). Skill: Microservices 0.9.',
        'human_score': 4,
    },
    {
        'prompt': 'Who collaborates most with Alice Chen?',
        'reference': 'Bob Kumar — collaboration weight 0.93.',
        'context': 'collaborations(p_alice, p_bob, weight=0.93). No other strong edges.',
        'human_score': 5,
    },
]

eval_df = pd.DataFrame(eval_cases)
print(f'✓ eval dataset: {len(eval_df)} rows')
eval_df

## Step 3 — Generate candidate responses

EvalTask can either receive a pre-computed `response` column or take a `model` argument and
run inference itself. We do the former so we can stream the same generations into both the
pointwise and pairwise tasks below.

In [ ]:
from google import genai

# google-genai client; auto-discovers Vertex AI from env (VERTEXAI=true) or explicit args.
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)


def _generate(model_id: str, prompt: str, context: str) -> str:
    resp = genai_client.models.generate_content(
        model=model_id,
        contents=[
            f'Use only the context below to answer.\n\n'
            f'CONTEXT:\n{context}\n\n'
            f'QUESTION: {prompt}'
        ],
    )
    return resp.text or ''


eval_df['response'] = [
    _generate(REASONING_MODEL, row.prompt, row.context) for row in eval_df.itertuples()
]
eval_df['baseline_response'] = [
    _generate(BASELINE_MODEL, row.prompt, row.context) for row in eval_df.itertuples()
]

print(f'✓ generated {len(eval_df)} candidate + baseline responses')
for i, row in eval_df.iterrows():
    print(f'\n[{i}] {row.prompt[:50]}')
    print(f'   candidate: {row.response[:100]}')
    print(f'   baseline:  {row.baseline_response[:100]}')

## Step 4 — Pointwise eval (groundedness + helpfulness)

Both metrics are pre-built in `MetricPromptTemplateExamples.Pointwise`. They use a
`gemini-2.5-flash` judge under the hood; cost ≈ \$0.001 per case.

EvalTask reads four columns by default:

| Column | Meaning |
|---|---|
| `prompt` | user input |
| `response` | candidate answer (what we're grading) |
| `reference` | gold answer (for metrics that need it) |
| `context` | grounding snippets (for groundedness) |

In [ ]:
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

pointwise_metrics = [
    MetricPromptTemplateExamples.Pointwise.GROUNDEDNESS,
    MetricPromptTemplateExamples.Pointwise.HELPFULNESS,
]

pointwise_task = EvalTask(
    dataset=eval_df,
    metrics=pointwise_metrics,
    experiment=f'{PREFIX}-pointwise',
)

pointwise_result = pointwise_task.evaluate(
    experiment_run_name=f'{PREFIX}-pointwise-{int(time.time())}'
)

print('✓ pointwise eval complete')
print('\n— summary metrics —')
print(json.dumps(pointwise_result.summary_metrics, indent=2, default=str))
print('\n— per-row scores —')
score_cols = [c for c in pointwise_result.metrics_table.columns if c.endswith('/score')]
display_cols = ['prompt'] + score_cols
pointwise_result.metrics_table[display_cols]

## Step 5 — Pairwise SxS (candidate vs baseline)

Pairwise metrics compare two responses side-by-side using a judge. We use the prebuilt
`Pairwise.QUESTION_ANSWERING_QUALITY` so the judge sees both `response` (candidate) and
`baseline_model_response` (baseline) columns.

EvalTask convention: pairwise metrics read the candidate from `response` and the baseline
from `baseline_model_response`. Rename the column we generated earlier.

In [ ]:
from vertexai.evaluation import PairwiseMetric

pairwise_df = eval_df.rename(columns={'baseline_response': 'baseline_model_response'})

pairwise_metric = PairwiseMetric(
    metric='pairwise_question_answering_quality',
    metric_prompt_template=MetricPromptTemplateExamples.Pairwise.QUESTION_ANSWERING_QUALITY.metric_prompt_template,
)

pairwise_task = EvalTask(
    dataset=pairwise_df,
    metrics=[pairwise_metric],
    experiment=f'{PREFIX}-pairwise',
)

pairwise_result = pairwise_task.evaluate(
    experiment_run_name=f'{PREFIX}-pairwise-{int(time.time())}'
)

print('✓ pairwise eval complete')
print('\n— summary metrics —')
print(json.dumps(pairwise_result.summary_metrics, indent=2, default=str))
print('\n— per-row preference —')
pref_cols = [c for c in pairwise_result.metrics_table.columns
             if 'pairwise_choice' in c or c.endswith('/explanation')]
pairwise_result.metrics_table[['prompt'] + pref_cols].head()

## Step 6 — Compare candidate vs human scores

If `human_score` is present we can sanity-check the auto-eval against the hand-graded
ground truth. This is what AWS-side Bedrock Online Evaluation does via its
`Correctness` evaluator.

In [ ]:
comparison = pointwise_result.metrics_table[
    ['prompt', 'groundedness/score', 'helpfulness/score']
].copy()
comparison['human_score'] = eval_df['human_score'].values
comparison['auto_avg'] = (
    comparison['groundedness/score'] + comparison['helpfulness/score']
) / 2.0
comparison['delta_vs_human'] = comparison['auto_avg'] - comparison['human_score']
comparison

## Step 7 — Persist experiment ids into config.json

In [ ]:
config = json.loads(Path('config.json').read_text())
config.setdefault('gcp', {}).setdefault('eval', {})
config['gcp']['eval']['pointwise_experiment'] = f'{PREFIX}-pointwise'
config['gcp']['eval']['pairwise_experiment'] = f'{PREFIX}-pairwise'
config['gcp']['eval']['last_run_ts'] = int(time.time())
config['gcp']['eval']['candidate_model'] = REASONING_MODEL
config['gcp']['eval']['baseline_model'] = BASELINE_MODEL
Path('config.json').write_text(json.dumps(config, indent=2))
print('✓ config.gcp.eval written')

## Summary

- 5-row pandas EvalDataset built with prompt / reference / context / human_score.
- Candidate (`gemini-2.5-pro`) and baseline (`gemini-2.5-flash`) responses generated via
  the unified `google-genai` SDK against Vertex AI.
- Pointwise eval ran `MetricPromptTemplateExamples.Pointwise.GROUNDEDNESS` and
  `.HELPFULNESS` through `EvalTask`.
- Pairwise SxS ran `Pairwise.QUESTION_ANSWERING_QUALITY` between candidate and baseline.
- Auto-eval scores compared against the `human_score` ground truth.
- Experiment ids persisted to `config.gcp.eval` so 17-cloud-trace can join eval scores
  against trace spans.

**Next:** [17-cloud-trace.ipynb](17-cloud-trace.ipynb) — Cloud Trace + OpenInference span projection.